# 05 - Texte : Optimisation et Modèles Alternatifs

## Objectif de ce notebook
**Améliorer** la baseline du notebook 04 en testant : rééquilibrage des classes, optimisation des hyperparamètres et modèles de type boosting (XGBoost, LightGBM, CatBoost).

**Prérequis** : Exécuter le notebook 04 pour disposer des modèles baseline, du vectoriseur TF-IDF et du label encoder dans `models/`.

**Livrable** : Meilleur modèle après optimisation (ex. SVM optimisé ou CatBoost), sauvegardé pour le notebook 06.

---

## Plan
1. Chargement des données et du meilleur modèle baseline (notebook 04)
2. Analyse approfondie des métriques par classe (top/bottom performances)
3. Optimisation des hyperparamètres (GridSearch/RandomSearch sur SVM)
4. Gestion du déséquilibre (class weights, SMOTE)
5. Modèles avancés (XGBoost, LightGBM, CatBoost)
6. Comparaison complète de tous les modèles
7. Sélection et sauvegarde du meilleur modèle



In [ ]:
# Import des bibliothèques nécessaires
import os
# Maximiser le parallélisme CPU (OpenMP, numpy, etc.) - avant les imports sklearn
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count() or 4)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
import pickle
warnings.filterwarnings('ignore')

# Ajouter le dossier src au path
sys.path.append(str(Path('../').resolve()))

# Import des modules
from src.utils.data_loader import load_data
from src.modeling import TFIDFVectorizer, BaselineModels, AdvancedModels
from src.optimization import (
    optimize_model,
    grid_search_optimization,
    random_search_optimization,
    create_class_weights,
    apply_smote,
    apply_adasyn
)
from src.evaluation import (
    evaluate_model,
    print_classification_report,
    plot_confusion_matrix,
    plot_class_distribution
)

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

# Configuration des graphiques
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Bibliothèques importées avec succès")



## 1. Chargement des Données et du Modèle Baseline

Chargement des données préprocessées et du meilleur modèle baseline de Step 1.


In [ ]:
# Chemins
DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')

# Chargement des données
print("🔄 Chargement des données...")
X_train = pd.read_csv(DATA_DIR / 'X_train_clean.csv')
X_test = pd.read_csv(DATA_DIR / 'X_test_clean.csv')
y_train = pd.read_csv(DATA_DIR / 'y_train.csv')

# Chargement optionnel : y_train_superclass (24 classes)
y_train_superclass = None
y_train_superclass_path = DATA_DIR / 'y_train_superclass.csv'
if y_train_superclass_path.exists():
    y_train_superclass = pd.read_csv(y_train_superclass_path)
else:
    # Recréer la superclasse si le fichier n'existe pas (Publications -> 9999)
    CODE_SUPERCLASSE = 9999
    PUBLICATIONS_CLASSES = {10, 2280, 2403, 2705}
    y_train_superclass = y_train.copy()
    y_train_superclass['prdtypecode'] = y_train['prdtypecode'].apply(
        lambda x: CODE_SUPERCLASSE if x in PUBLICATIONS_CLASSES else x
    )
    y_train_superclass.to_csv(y_train_superclass_path, index=False)
    print("⚠️ y_train_superclass.csv introuvable : superclasse recréée et sauvegardée.")

# Chargement du vectoriseur et des label encoders
print("🔄 Chargement du vectoriseur et des label encoders...")
vectorizer = TFIDFVectorizer.load(MODELS_DIR / 'tfidf_vectorizer.pkl')
with open(MODELS_DIR / 'label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

label_encoder_superclass = None
label_encoder_superclass_path = MODELS_DIR / 'label_encoder_superclass.pkl'
if label_encoder_superclass_path.exists():
    with open(label_encoder_superclass_path, 'rb') as f:
        label_encoder_superclass = pickle.load(f)

# Vectorisation
X_train_texts = X_train['text_combined'].astype(str)
X_test_texts = X_test['text_combined'].astype(str)
X_train_vect = vectorizer.transform(X_train_texts)
X_test_vect = vectorizer.transform(X_test_texts)

# Encodage des labels
y_train_labels = y_train['prdtypecode'].values
y_train_encoded = label_encoder.transform(y_train_labels)

y_train_encoded_sc = None
if y_train_superclass is not None:
    if label_encoder_superclass is None:
        label_encoder_superclass = LabelEncoder()
        label_encoder_superclass.fit(y_train_superclass['prdtypecode'].values)
        with open(label_encoder_superclass_path, 'wb') as f:
            pickle.dump(label_encoder_superclass, f)
    y_train_encoded_sc = label_encoder_superclass.transform(
        y_train_superclass['prdtypecode'].values
    )

# Split indices (mêmes indices pour tous les scénarios)
indices = np.arange(X_train_vect.shape[0])
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y_train_encoded
)

X_train_split = X_train_vect[train_idx]
X_val_split = X_train_vect[val_idx]

# Scénarios
SCENARIOS = [("27 classes", y_train_encoded, label_encoder)]
if y_train_encoded_sc is not None:
    SCENARIOS.append(("24 classes (superclass)", y_train_encoded_sc, label_encoder_superclass))

# Construire les splits par scénario
splits_by_scenario = {}
for scenario_name, y_encoded, _ in SCENARIOS:
    y_train_split = y_encoded[train_idx]
    y_val_split = y_encoded[val_idx]
    splits_by_scenario[scenario_name] = (X_train_split, X_val_split, y_train_split, y_val_split)

print(f"✅ Données chargées et préparées !")
print(f"  - Train : {X_train_split.shape[0]:,} échantillons")
print(f"  - Validation : {X_val_split.shape[0]:,} échantillons")
print(f"  - Test : {X_test_vect.shape[0]:,} échantillons")
print(f"  - Vocabulaire : {vectorizer.get_vocabulary_size():,} mots")
print(f"✅ Scénarios à tester : {[s[0] for s in SCENARIOS]}")



## 2. Analyse Approfondie des Métriques Baseline

Analyse détaillée des performances du modèle baseline pour identifier les axes d'amélioration.


In [ ]:
# Chargement du meilleur modèle baseline (SVM Linear) par scénario
print("🔄 Chargement du modèle baseline...")

baseline_models_by_scenario = {}
metrics_baseline_by_scenario = {}
y_pred_baseline_by_scenario = {}

from sklearn.metrics import classification_report

for scenario_name, _, _ in SCENARIOS:
    scenario_suffix = "superclass" if "superclass" in scenario_name.lower() else "27_classes"
    X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario_name]

    print("="*80)
    print(f"PERFORMANCES BASELINE (SVM Linear) - {scenario_name}")
    print("="*80)

    # Chercher le fichier du modèle baseline
    baseline_files = list(MODELS_DIR.glob(f"*{scenario_suffix}_baseline.pkl"))
    svm_files = [f for f in baseline_files if 'svm' in f.name.lower()]
    if svm_files:
        baseline_model = BaselineModels.load_model(svm_files[0])
        print(f"✅ Modèle baseline chargé : {svm_files[0].name}")
    elif baseline_files:
        baseline_model = BaselineModels.load_model(baseline_files[0])
        print(f"✅ Modèle baseline chargé : {baseline_files[0].name}")
    else:
        # Si pas de modèle sauvegardé, réentraîner rapidement
        print("⚠️  Modèle baseline non trouvé, réentraînement rapide...")
        baseline_model = LinearSVC(random_state=42, dual=False, max_iter=2000)
        import time
        _t0 = time.time()
        baseline_model.fit(X_train_split, y_train_split)
        print(f"Temps d'entraînement : {time.time() - _t0} secondes")
        print("✅ Modèle baseline réentraîné")

    # Prédictions baseline
    y_pred_baseline = baseline_model.predict(X_val_split)

    # Évaluation baseline
    metrics_baseline = evaluate_model(y_val_split, y_pred_baseline)

    baseline_models_by_scenario[scenario_name] = baseline_model
    metrics_baseline_by_scenario[scenario_name] = metrics_baseline
    y_pred_baseline_by_scenario[scenario_name] = y_pred_baseline

    print(f"  - Accuracy : {metrics_baseline['accuracy']:.4f}")
    print(f"  - F1-score (macro) : {metrics_baseline['f1_macro']:.4f}")
    print(f"  - F1-score (weighted) : {metrics_baseline['f1_weighted']:.4f}")
    print(f"  - Precision (macro) : {metrics_baseline['precision_macro']:.4f}")
    print(f"  - Recall (macro) : {metrics_baseline['recall_macro']:.4f}")

    # Analyse par classe
    report = classification_report(y_val_split, y_pred_baseline, output_dict=True, zero_division=0)
    df_report = pd.DataFrame(report).transpose()

    print(f"\n📊 Performances par classe (top 5 meilleures et pires) :")
    print("\nTop 5 meilleures classes :")
    print(df_report.nlargest(6, 'f1-score').head(5)[['precision', 'recall', 'f1-score', 'support']])

    print("\nTop 5 pires classes :")
    print(df_report.nsmallest(6, 'f1-score').tail(5)[['precision', 'recall', 'f1-score', 'support']])
    print("\n")



## 3. Optimisation des Hyperparamètres

Optimisation des hyperparamètres du meilleur modèle baseline (SVM Linear) avec GridSearch et RandomSearch.


In [ ]:
# Optimisation de SVM Linear avec RandomSearch (plus rapide que GridSearch)
print("="*80)
print("OPTIMISATION SVM LINEAR")
print("="*80)

# Grille de paramètres pour RandomSearch
svm_param_dist = {
    'C': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    'penalty': ['l2'],
    'loss': ['squared_hinge'],
    'max_iter': [1000, 2000, 3000],
    'dual': [False]  # Pour datasets larges
}

svm_optimized_by_scenario = {}
metrics_svm_opt_by_scenario = {}
y_pred_svm_opt_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario_name]
    metrics_baseline = metrics_baseline_by_scenario[scenario_name]

    print(f"\n🔄 Optimisation SVM Linear - {scenario_name}...")

    # Création du modèle
    svm_model = LinearSVC(random_state=42, dual=False)

    # Optimisation
    svm_optimized = random_search_optimization(
        svm_model,
        X_train_split,
        y_train_split,
        svm_param_dist,
        n_iter=20,  # Nombre de combinaisons à tester
        cv=3,  # 3 folds pour accélérer
        n_jobs=-1,
        random_state=42
    )

    # Évaluation du modèle optimisé
    y_pred_svm_opt = svm_optimized['best_model'].predict(X_val_split)
    metrics_svm_opt = evaluate_model(y_val_split, y_pred_svm_opt)

    svm_optimized_by_scenario[scenario_name] = svm_optimized
    metrics_svm_opt_by_scenario[scenario_name] = metrics_svm_opt
    y_pred_svm_opt_by_scenario[scenario_name] = y_pred_svm_opt

    print(f"\n📊 Comparaison SVM Baseline vs Optimisé ({scenario_name}) :")
    print(f"  Baseline - F1-macro : {metrics_baseline['f1_macro']:.4f}")
    print(f"  Optimisé  - F1-macro : {metrics_svm_opt['f1_macro']:.4f}")
    print(f"  Amélioration : {(metrics_svm_opt['f1_macro'] - metrics_baseline['f1_macro']) / metrics_baseline['f1_macro'] * 100:+.2f}%")



## 4. Gestion du Déséquilibre de Classes

Application de techniques de rééquilibrage : class weights et SMOTE.


In [ ]:
# Création des class weights
print("="*80)
print("CRÉATION DES CLASS WEIGHTS")
print("="*80)

class_weights_by_scenario = {}
svm_weighted_by_scenario = {}
metrics_svm_weighted_by_scenario = {}
y_pred_svm_weighted_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario_name]
    metrics_baseline = metrics_baseline_by_scenario[scenario_name]

    print(f"\n🔄 Test SVM avec class weights - {scenario_name}...")
    class_weights = create_class_weights(y_train_split, method='balanced')
    class_weights_by_scenario[scenario_name] = class_weights

    svm_weighted = LinearSVC(
        C=1.0,
        class_weight=class_weights,
        random_state=42,
        dual=False,
        max_iter=2000
    )
    import time
    start_time = time.time()
    svm_weighted.fit(X_train_split, y_train_split)
    print(f"Temps d'entraînement : {time.time() - start_time} secondes")
    y_pred_svm_weighted = svm_weighted.predict(X_val_split)
    metrics_svm_weighted = evaluate_model(y_val_split, y_pred_svm_weighted)

    svm_weighted_by_scenario[scenario_name] = svm_weighted
    metrics_svm_weighted_by_scenario[scenario_name] = metrics_svm_weighted
    y_pred_svm_weighted_by_scenario[scenario_name] = y_pred_svm_weighted

    print(f"\n📊 Comparaison ({scenario_name}) :")
    print(f"  Baseline (sans weights) - F1-macro : {metrics_baseline['f1_macro']:.4f}")
    print(f"  Avec class weights      - F1-macro : {metrics_svm_weighted['f1_macro']:.4f}")
    print(f"  Amélioration : {(metrics_svm_weighted['f1_macro'] - metrics_baseline['f1_macro']) / metrics_baseline['f1_macro'] * 100:+.2f}%")



In [ ]:
# Test avec SMOTE
import time
print("="*80)
print("TEST AVEC SMOTE")
print("="*80)
print("SMOTE génère des échantillons synthétiques pour les classes minoritaires.\n")

metrics_ref_by_scenario = {}
metrics_svm_smote_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario_name]

    # Échantillon pour accélérer (SMOTE + matrices denses = coût mémoire)
    n_samples = X_train_split.shape[0]
    sample_size = min(15000, n_samples)
    np.random.seed(42)
    indices = np.random.choice(n_samples, sample_size, replace=False)
    X_train_sample = X_train_split[indices]
    y_train_sample = y_train_split[indices]

    # SMOTE nécessite des matrices denses (ou sparse selon version imblearn)
    from scipy.sparse import issparse
    if issparse(X_train_sample):
        print(f"Conversion en matrice dense pour SMOTE ({sample_size} × {X_train_sample.shape[1]} features) - {scenario_name}...")
        X_train_sample = X_train_sample.toarray()

    print(f"Échantillon : {sample_size} échantillons, {len(np.unique(y_train_sample))} classes ({scenario_name})\n")

    # Référence : SVM sur le même échantillon SANS SMOTE (comparaison équitable)
    print(f"🔄 Référence : SVM sur échantillon sans SMOTE ({scenario_name})...")
    svm_ref = LinearSVC(C=1.0, random_state=42, dual=False, max_iter=2000)
    svm_ref.fit(X_train_sample, y_train_sample)
    y_pred_ref = svm_ref.predict(X_val_split)
    metrics_ref = evaluate_model(y_val_split, y_pred_ref)
    metrics_ref_by_scenario[scenario_name] = metrics_ref
    print(f"  F1-macro (sans SMOTE) : {metrics_ref['f1_macro']:.4f}\n")

    # Application de SMOTE (k_neighbors=3 pour classes très minoritaires)
    X_train_smote, y_train_smote = apply_smote(
        X_train_sample,
        y_train_sample,
        sampling_strategy='auto',
        random_state=42,
        k_neighbors=3
    )

    # Entraînement SVM sur données SMOTE
    print(f"\n🔄 Entraînement SVM avec données SMOTE ({scenario_name})...")
    svm_smote = LinearSVC(C=1.0, random_state=42, dual=False, max_iter=2000)
    _t0 = time.time()
    svm_smote.fit(X_train_smote, y_train_smote)
    print(f"Temps d'entraînement : {time.time() - _t0:.1f} secondes")

    # Évaluation sur le jeu de validation (inchangé)
    y_pred_svm_smote = svm_smote.predict(X_val_split)
    metrics_svm_smote = evaluate_model(y_val_split, y_pred_svm_smote)
    metrics_svm_smote_by_scenario[scenario_name] = metrics_svm_smote

    print(f"\n📊 Comparaison ({scenario_name}) :")
    print(f"  Baseline (sans SMOTE) - F1-macro : {metrics_ref['f1_macro']:.4f}")
    print(f"  Avec SMOTE            - F1-macro : {metrics_svm_smote['f1_macro']:.4f}")
    diff_pct = (metrics_svm_smote['f1_macro'] - metrics_ref['f1_macro']) / metrics_ref['f1_macro'] * 100
    print(f"  Amélioration : {diff_pct:+.2f}%\n")



## 5. Modèles Avancés

Test de modèles plus complexes : XGBoost, LightGBM, CatBoost, Random Forest optimisé.


In [ ]:
# Création et entraînement des modèles avancés
print("="*80)
print("MODÈLES AVANCÉS")
print("="*80)

advanced_models_by_scenario = {}
trained_advanced_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario_name]
    class_weights = class_weights_by_scenario.get(scenario_name)

    print(f"\n📋 Modèles avancés - {scenario_name}")
    advanced_models = AdvancedModels(random_state=42)
    models_advanced = advanced_models.create_advanced_models(class_weights=class_weights)

    print(f"  - {len(models_advanced)} modèles créés")
    for model_name in models_advanced.keys():
        print(f"    • {model_name}")

    print(f"\n🔄 Entraînement des modèles avancés ({scenario_name})...")
    print("="*80)

    trained_advanced = advanced_models.train_all_models(X_train_split, y_train_split)

    advanced_models_by_scenario[scenario_name] = advanced_models
    trained_advanced_by_scenario[scenario_name] = trained_advanced

    print("\n" + "="*80)
    print(f"✅ Modèles avancés entraînés ({scenario_name}) !")



## 6. Comparaison Complète de Tous les Modèles

Évaluation et comparaison de tous les modèles testés.


In [ ]:
# Évaluation de tous les modèles
print("="*80)
print("COMPARAISON COMPLÈTE DES MODÈLES")
print("="*80)

all_results = []

for scenario_name, _, _ in SCENARIOS:
    X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario_name]
    metrics_baseline = metrics_baseline_by_scenario[scenario_name]
    metrics_svm_opt = metrics_svm_opt_by_scenario[scenario_name]
    metrics_svm_weighted = metrics_svm_weighted_by_scenario[scenario_name]

    # Baseline
    all_results.append({
        'Scenario': scenario_name,
        'Model': 'SVM Linear (Baseline)',
        'Accuracy': metrics_baseline['accuracy'],
        'F1_macro': metrics_baseline['f1_macro'],
        'F1_weighted': metrics_baseline['f1_weighted'],
        'Precision_macro': metrics_baseline['precision_macro'],
        'Recall_macro': metrics_baseline['recall_macro']
    })

    # SVM Optimisé
    all_results.append({
        'Scenario': scenario_name,
        'Model': 'SVM Linear (Optimized)',
        'Accuracy': metrics_svm_opt['accuracy'],
        'F1_macro': metrics_svm_opt['f1_macro'],
        'F1_weighted': metrics_svm_opt['f1_weighted'],
        'Precision_macro': metrics_svm_opt['precision_macro'],
        'Recall_macro': metrics_svm_opt['recall_macro']
    })

    # SVM avec class weights
    all_results.append({
        'Scenario': scenario_name,
        'Model': 'SVM Linear (Class Weights)',
        'Accuracy': metrics_svm_weighted['accuracy'],
        'F1_macro': metrics_svm_weighted['f1_macro'],
        'F1_weighted': metrics_svm_weighted['f1_weighted'],
        'Precision_macro': metrics_svm_weighted['precision_macro'],
        'Recall_macro': metrics_svm_weighted['recall_macro']
    })

    # Modèles avancés
    advanced_models = advanced_models_by_scenario[scenario_name]
    trained_advanced = trained_advanced_by_scenario[scenario_name]
    for model_name in trained_advanced.keys():
        y_pred = advanced_models.predict(model_name, X_val_split)
        metrics = evaluate_model(y_val_split, y_pred)
        all_results.append({
            'Scenario': scenario_name,
            'Model': model_name,
            'Accuracy': metrics['accuracy'],
            'F1_macro': metrics['f1_macro'],
            'F1_weighted': metrics['f1_weighted'],
            'Precision_macro': metrics['precision_macro'],
            'Recall_macro': metrics['recall_macro']
        })

# DataFrame de comparaison
results_comparison = pd.DataFrame(all_results).sort_values(['Scenario', 'F1_macro'], ascending=[True, False])

print("\n📊 Résultats comparatifs (triés par F1-macro) :")
print("="*80)
print(results_comparison.to_string(index=False))

# Tableaux séparés par scénario
for scenario in results_comparison['Scenario'].unique():
    print(f"\n{'-'*80}")
    print(f"TABLEAU RÉSULTATS - {scenario}")
    print(f"{'-'*80}")
    scenario_df = results_comparison[results_comparison['Scenario'] == scenario]
    print(scenario_df.to_string(index=False))



In [ ]:
# Visualisation de la comparaison (par scénario)
for scenario in results_comparison['Scenario'].unique():
    scenario_df = results_comparison[results_comparison['Scenario'] == scenario]

    y_pos = np.arange(len(scenario_df))
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # F1-score macro
    axes[0].barh(y_pos, scenario_df['F1_macro'].values)
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(scenario_df['Model'])
    axes[0].set_xlabel('F1-score (macro)', fontsize=12)
    axes[0].set_title(f'F1-score Macro par Modèle - {scenario}', fontsize=14)
    axes[0].grid(axis='x', alpha=0.3)
    axes[0].invert_yaxis()

    # F1-score weighted
    axes[1].barh(y_pos, scenario_df['F1_weighted'].values, color='green')
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(scenario_df['Model'])
    axes[1].set_xlabel('F1-score (weighted)', fontsize=12)
    axes[1].set_title(f'F1-score Weighted par Modèle - {scenario}', fontsize=14)
    axes[1].grid(axis='x', alpha=0.3)
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    # Graphique comparatif complet
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(scenario_df))
    width = 0.2

    ax.bar(x - width*1.5, scenario_df['Accuracy'].values, width, label='Accuracy', alpha=0.8)
    ax.bar(x - width*0.5, scenario_df['F1_macro'].values, width, label='F1-score (macro)', alpha=0.8)
    ax.bar(x + width*0.5, scenario_df['F1_weighted'].values, width, label='F1-score (weighted)', alpha=0.8)
    ax.bar(x + width*1.5, scenario_df['Precision_macro'].values, width, label='Precision (macro)', alpha=0.8)

    ax.set_xlabel('Modèles', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'Comparaison Complète des Métriques - {scenario}', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(scenario_df['Model'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()



## 7. Sélection du Meilleur Modèle

Identification et analyse du meilleur modèle final.


In [ ]:
# Identification du meilleur modèle par scénario
best_by_scenario = {}

print("="*80)
print("MEILLEUR MODÈLE IDENTIFIÉ")
print("="*80)

for scenario in results_comparison['Scenario'].unique():
    scenario_df = results_comparison[results_comparison['Scenario'] == scenario]
    best_row = scenario_df.iloc[0]
    best_model_name = best_row['Model']
    best_f1_macro = best_row['F1_macro']
    best_accuracy = best_row['Accuracy']

    print(f"🏆 Modèle : {best_model_name} ({scenario})")
    print(f"   F1-score (macro) : {best_f1_macro:.4f}")
    print(f"   F1-score (weighted) : {best_row['F1_weighted']:.4f}")
    print(f"   Accuracy : {best_accuracy:.4f}")
    print(f"\n📈 Amélioration vs Baseline :")

    baseline_f1 = metrics_baseline_by_scenario[scenario]['f1_macro']
    improvement = (best_f1_macro - baseline_f1) / baseline_f1 * 100
    print(f"   Baseline F1-macro : {baseline_f1:.4f}")
    print(f"   Meilleur F1-macro : {best_f1_macro:.4f}")
    print(f"   Amélioration : {improvement:+.2f}%")

    # Récupération du meilleur modèle
    if best_model_name == 'SVM Linear (Baseline)':
        best_model = baseline_models_by_scenario[scenario]
        y_pred_best = y_pred_baseline_by_scenario[scenario]
    elif best_model_name == 'SVM Linear (Optimized)':
        best_model = svm_optimized_by_scenario[scenario]['best_model']
        y_pred_best = y_pred_svm_opt_by_scenario[scenario]
    elif best_model_name == 'SVM Linear (Class Weights)':
        best_model = svm_weighted_by_scenario[scenario]
        y_pred_best = y_pred_svm_weighted_by_scenario[scenario]
    else:
        # Modèle avancé
        advanced_models = advanced_models_by_scenario[scenario]
        best_model = advanced_models.trained_models[best_model_name]
        X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario]
        y_pred_best = advanced_models.predict(best_model_name, X_val_split)

    # Label encoder du scénario
    label_encoder_best = next(enc for name, _, enc in SCENARIOS if name == scenario)
    class_names = [f"Classe {cls}" for cls in sorted(label_encoder_best.classes_)]

    # Rapport de classification détaillé
    print(f"\n{'='*80}")
    print(f"RAPPORT DE CLASSIFICATION DÉTAILLÉ - {best_model_name} ({scenario})")
    print(f"{'='*80}")
    X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario]
    print_classification_report(y_val_split, y_pred_best, target_names=class_names)
    print("\n")

    best_by_scenario[scenario] = {
        'model_name': best_model_name,
        'model': best_model,
        'y_pred': y_pred_best,
        'class_names': class_names
    }



### 7.1 Matrice de Confusion du Meilleur Modèle


In [ ]:
# Matrices de confusion par scénario
for scenario, data in best_by_scenario.items():
    X_train_split, X_val_split, y_train_split, y_val_split = splits_by_scenario[scenario]
    plot_confusion_matrix(
        y_val_split,
        data['y_pred'],
        class_names=data['class_names'],
        figsize=(20, 16),
        normalize=True
    )
    print(f"Matrice de confusion - {data['model_name']} ({scenario})")



## 8. Sauvegarde du Meilleur Modèle

Sauvegarde du modèle final pour utilisation future.


In [ ]:
# Sauvegarde des meilleurs modèles par scénario
print("💾 Sauvegarde du meilleur modèle...")

for scenario, data in best_by_scenario.items():
    scenario_suffix = "superclass" if "superclass" in scenario.lower() else "27_classes"
    model_name = data['model_name']
    model_filename = model_name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace(',', '')
    model_path = MODELS_DIR / f'{model_filename}_{scenario_suffix}_final.pkl'

    advanced_models = advanced_models_by_scenario[scenario]
    if model_name in advanced_models.trained_models:
        advanced_models.save_model(model_name, model_path)
    else:
        with open(model_path, 'wb') as f:
            pickle.dump(data['model'], f)

    print(f"✅ Modèle sauvegardé : {model_path}")

# Sauvegarde des résultats de comparaison
results_path = MODELS_DIR / 'model_comparison_results.csv'
results_comparison.to_csv(results_path, index=False)
print(f"✅ Résultats de comparaison sauvegardés : {results_path}")



## 9. Synthèse et Conclusions

### Résumé des Résultats

- ✅ Modèles baseline analysés et optimisés
- ✅ Techniques de rééquilibrage testées (class weights)
- ✅ Modèles avancés testés (XGBoost, LightGBM, CatBoost, etc.)
- ✅ Meilleur modèle identifié : **[sera rempli après exécution]**
- ✅ Amélioration vs baseline : **[sera rempli après exécution]**

### Prochaines Étapes → Notebook 06

1. **Modélisation avancée** : Ensembles (Voting, Stacking), Deep Learning (MLP, CNN, LSTM)
2. **Interprétabilité** : SHAP, feature importance pour comprendre les prédictions
3. **Conclusions scientifiques et métiers** : Analyser le succès ou l'échec de la modélisation

### Fichiers Générés

- `models/[model]_final.pkl` : Meilleur modèle final
- `models/model_comparison_results.csv` : Résultats de comparaison de tous les modèles

